0111 26

alert app was hacked
link to tg channel
a lot of presents with adds

so i-parse

!! setup yor own telegram api creds

#setup

In [1]:
!pip install telethon

In [2]:
# get on my.telegrom.org - create app

# api_id = "api_id"
# api_hash = ""
session_name = 'my_session'

""" or colab secrets
from google.colab import userdata
api_id = userdata.get('TELEGRAM_API_ID')
api_hash = userdata.get('TELEGRAM_API_HASH')
"""

I got a problem with auth by code? so decided use qr

In [3]:
!pip install qrcode

In [4]:
from telethon import TelegramClient
import qrcode
import asyncio

async def main():
    client = TelegramClient(session_name, api_id, api_hash)
    await client.connect()

    if await client.is_user_authorized():
        print("user_authorized")
        await client.disconnect()
        return

    qr_login = await client.qr_login()

    qr = qrcode.QRCode()
    qr.add_data(qr_login.url)
    qr.print_ascii()
    print("scan at teleram - add devie")

    try:
        await qr_login.wait()
        print("login successful")
    except asyncio.TimeoutError:
        print("too long, try again")
    except Exception as e:
        if "SessionPasswordNeededError" in str(type(e)):
            password = input("2fa_pwd: ")
            await client.sign_in(password=password)
            print("login successful")
        else:
            print(f"error: {e}")

    await client.disconnect()

In [5]:
await main()

Вже авторизовано


In [6]:
async def find_user_by_username(client, username):
    try:
        clean_username = username.strip().replace('@', '')
        entity = await client.get_entity(clean_username)

        print("founded ")
        print(f"ID: {entity.id}")

        if hasattr(entity, 'first_name'):
            print(f"first_name: {entity.first_name} {entity.last_name or ''}")
        elif hasattr(entity, 'title'):
            print(f"title: {entity.title}")
        if hasattr(entity, 'username') and entity.username:
            print(f" @{entity.username}")

        print(f"type: {type(entity).__name__}")

        return entity

    except Exception as e:
        print(f"\not faund  '{username}'.")
        print(f"error {e}")
        return None

In [7]:
client = TelegramClient(session_name, api_id, api_hash)
await client.start()

# user_input = input("nikname:")
user_input = "hitcorecorp"

user_entity = await find_user_by_username(client, user_input)

founded 
ID: 4482571876
title: Hitcore
 @hitcorecorp
type: Channel


# get user gifts


In [8]:
from telethon.tl.functions.payments import GetSavedStarGiftsRequest

async def get_user_gifts(client, user_entity):
    try:
        result = await client(GetSavedStarGiftsRequest(
            peer=user_entity,
            offset='0',
            exclude_unsaved=True,
            limit=1000
        ))

        print(f"find presents: {len(result.gifts)}")
        for gift in result.gifts:
            print(gift)
        return result.gifts
    except Exception as e:
        print(f"error: {e}")

In [9]:
# await get_user_gifts(client, user_entity)

In [ ]:
import pickle

gifts_list = await get_user_gifts(client, user_entity)

if gifts_list:
    filename = "my_gifts.pkl"
    with open(filename, 'wb') as f:
      pickle.dump(gifts_list, f)
    print(f"\n saved {len(gifts_list)} presents to file '{filename}'")
else:
    print("error")



010  - saved 100 gifts


In [13]:
print(gifts_list[0])

SavedStarGift(date=datetime.datetime(2026, 11, 1, 22, 14, 2, tzinfo=datetime.timezone.utc), gift=StarGift(id=5170145012310081615, sticker=Document(id=5465263910414195580, access_hash=-5361215607397896712, file_reference=b'\x00j\xe7\xb9\xc6\xb3\x11%Dh\xe6\x07\xeb\x126Q1\xadBGj', date=datetime.datetime(2022, 8, 16, 10, 59, 49, tzinfo=datetime.timezone.utc), mime_type='application/x-tgsticker', size=25528, dc_id=2, attributes=[DocumentAttributeImageSize(w=512, h=512), DocumentAttributeCustomEmoji(alt='💝', stickerset=InputStickerSetID(id=7173162320003080, access_hash=-1386065779200000200), free=False, text_color=False), DocumentAttributeFilename(file_name='AnimatedSticker.tgs')], thumbs=[PhotoPathSize(type='j', bytes=b'\x1a\x04\xb1\x06\xdc]\x81ocF\x07tHGnZ{WL\x83k\x9at\x92IH\x85c\x88m\x82I\x8b_\x83ZZ\x90c\xb7~\x88WheI\x04]N\x00\x82J\x86V\x8e\\\x81A\x86\x81\x86C\x81KGZHeFF\x05\x99M\x01\x89\x02N\t\xa9K\x88\t\x85\x8c\x01\x9f\x89\x88\x9c\xa6\x9f\xa6\x82\x80\x82C\x83D\x88L\x92W\x9ca\xaeh\x8c\x0

In [15]:
async def get_all_user_gifts(client, user_entity):
    all_gifts = []
    offset = '0'

    while True:
        try:
            result = await client(GetSavedStarGiftsRequest(
                peer=user_entity,
                offset=offset,
                exclude_unsaved=True,
                limit=100
            ))

            all_gifts.extend(result.gifts)
            print(f"current: {len(result.gifts)}, all: {len(all_gifts)}")

            if not result.next_offset:
                break

            offset = result.next_offset

        except Exception as e:
            print(f"error: {e}")
            break

    return all_gifts

In [16]:
gifts_list = await get_all_user_gifts(client, user_entity)

if gifts_list:
    filename = "all_gifts.pkl"
    with open(filename, 'wb') as f:
      pickle.dump(gifts_list, f)
    print(f"\n saved {len(gifts_list)} presents to file '{filename}'")
else:
    print("error")

current: 100, all: 100
current: 100, all: 200
current: 100, all: 300
current: 100, all: 400
current: 100, all: 500
current: 100, all: 600
current: 100, all: 700
current: 100, all: 800
current: 3, all: 803

 saved 803 presents to file 'all_gifts.pkl'


022

# analyze

In [ ]:
import pickle
from collections import Counter